# LeetCode 417 — Pacific Atlantic Water Flow bằng Python

Link đề: https://leetcode.com/problems/pacific-atlantic-water-flow/ · Độ khó: **Medium** · Chủ đề: Depth-First Search, Breadth-First Search, Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc. Thứ tự các ô trong kết quả **không quan trọng**, nên khi kiểm tra ta **sắp xếp** kết quả rồi so sánh. Mọi đáp án được đối chiếu với một oracle vét cạn độc lập (đi xuôi dòng nước từ từng ô).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một đảo hình chữ nhật `m x n` giáp **Thái Bình Dương** (cạnh trên và cạnh trái) và **Đại Tây Dương** (cạnh dưới và cạnh phải); nước chảy sang ô kề (4 hướng) có độ cao **nhỏ hơn hoặc bằng**. Tìm mọi ô mà nước từ đó chảy được tới **cả hai** đại dương.
2. Nắm công cụ Python: lưới 2 chiều, bốn hướng đi `(dx, dy)`, kiểm tra biên `0 <= x < m`, `collections.deque` (hàng đợi BFS), `list` làm stack (DFS lặp), `set`, giao tập hợp `&`, `sorted`.
3. Đi từ cách **đi xuôi từ từng ô (vét cạn)** → **đảo chiều: BFS ngược từ bờ biển** → **DFS lặp bằng stack** → **quy hoạch động theo độ cao tăng dần có xử lý "cao nguyên"**.
4. Hiểu mấu chốt: bài toán "ô nào chảy tới biển" khó, nhưng bài toán **ngược** "từ biển, leo ngược lên những ô cao hơn hoặc bằng" dễ — chỉ cần **hai lần duyệt đồ thị** từ các ô sát bờ rồi lấy **giao**.
5. Tránh các lỗi kinh điển: đi ngược nhưng so sánh sai chiều (`<=` thay vì `>=`), chỉ lấy ô góc làm điểm xuất phát, quên tập hợp tách biệt cho hai đại dương, đệ quy quá sâu gây `RecursionError`, đánh dấu `visited` muộn khiến hàng đợi phình ra.

## 2. Đề bài & ví dụ

Có một hòn đảo hình chữ nhật `m x n` giáp cả **Thái Bình Dương** và **Đại Tây Dương**. Thái Bình Dương chạm **cạnh trái và cạnh trên** của đảo; Đại Tây Dương chạm **cạnh phải và cạnh dưới**.

Đảo chia thành lưới các ô vuông. Cho ma trận `heights` cỡ `m x n`, `heights[r][c]` là **độ cao so với mực nước biển** của ô `(r, c)`.

Trời mưa lớn. Nước mưa chảy từ một ô sang ô kề (Bắc, Nam, Đông, Tây) **nếu độ cao ô kề nhỏ hơn hoặc bằng** độ cao ô hiện tại. Nước ở bất kỳ ô nào **kề đại dương** có thể chảy ra đại dương đó.

Trả về danh sách `result` gồm các tọa độ `[r, c]` mà nước mưa từ ô `(r, c)` có thể chảy tới **cả hai** đại dương.

**Ví dụ 1:** `heights = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]` → `[[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]`.

**Ví dụ 2:** `heights = [[1]]` → `[[0,0]]` (một ô vừa kề Thái Bình Dương vừa kề Đại Tây Dương).

**Ràng buộc:**

- `m == heights.length`, `n == heights[r].length`
- `1 <= m, n <= 200`
- `0 <= heights[r][c] <= 10^5`

In [1]:
from collections import deque
import random
import sys
import time
from typing import List

ex1 = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]
examples = [
    (ex1, [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]),
    ([[1]], [[0, 0]]),
]


def norm(res):
    """Thứ tự kết quả không quan trọng: chuẩn hóa bằng cách sắp xếp (và loại trùng để phát hiện lỗi trùng bằng độ dài)."""
    return sorted([list(p) for p in res])


for heights, expected in examples:
    print(f"lưới {len(heights)}x{len(heights[0])} -> {len(expected)} ô")

lưới 5x5 -> 7 ô
lưới 1x1 -> 1 ô


### 2.1. Đọc kỹ đề

- Nước chảy **xuống dốc hoặc đi ngang** (`<=`), không bao giờ leo lên. Ô cao bằng nhau **có** chảy qua lại được (cao nguyên).
- Ô **kề đại dương** là những ô ở mép tương ứng: hàng `0` và cột `0` kề Thái Bình Dương; hàng `m - 1` và cột `n - 1` kề Đại Tây Dương. **Ô góc** thuộc hai mép: ô `(0, n-1)` và `(m-1, 0)` kề cả hai đại dương.
- Một ô "chảy tới đại dương" nếu có **một đường đi** (dãy ô kề nhau, độ cao không tăng) từ nó đến một ô sát đại dương đó.
- Kết quả không cần theo thứ tự (đề không quy định); ta so sánh sau khi `sorted`.

## 3. Kiến thức nền

### 3.1. Lưới 2 chiều và bốn hướng

`heights[r][c]` là ô hàng `r`, cột `c`. Bốn hướng kề: lên `(-1, 0)`, phải `(0, 1)`, xuống `(1, 0)`, trái `(0, -1)`. Từ `(x, y)` ô kề là `(x + dx, y + dy)`. **Luôn kiểm tra biên** `0 <= nx < m` và `0 <= ny < n` trước khi truy cập `heights[nx][ny]` (nếu không, chỉ số âm của Python sẽ "quay vòng" và cho kết quả sai lặng lẽ).

In [2]:
DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
grid = [[1, 2], [3, 4]]
m, n = 2, 2
x, y = 0, 0
for dx, dy in DIRS:
    nx, ny = x + dx, y + dy
    ok = 0 <= nx < m and 0 <= ny < n
    print(f"từ ({x},{y}) đi ({dx:>2},{dy:>2}) -> ({nx:>2},{ny:>2}) hợp lệ? {ok}", grid[nx][ny] if ok else "")
print("cẩn thận: grid[-1][0] =", grid[-1][0], "(chỉ số âm quay vòng, không báo lỗi!)")

từ (0,0) đi (-1, 0) -> (-1, 0) hợp lệ? False 
từ (0,0) đi ( 0, 1) -> ( 0, 1) hợp lệ? True 2
từ (0,0) đi ( 1, 0) -> ( 1, 0) hợp lệ? True 3
từ (0,0) đi ( 0,-1) -> ( 0,-1) hợp lệ? False 
cẩn thận: grid[-1][0] = 3 (chỉ số âm quay vòng, không báo lỗi!)


### 3.2. Hàng đợi `deque` và BFS

BFS (duyệt theo chiều rộng) dùng **hàng đợi**: lấy phần tử ở **đầu** (`popleft`), thêm phần tử mới vào **cuối** (`append`). `collections.deque` làm cả hai việc trong `O(1)` (còn `list.pop(0)` tốn `O(n)`).

In [3]:
q = deque([(0, 0)])
q.append((0, 1))
q.append((1, 1))
print(q, "| lấy ra:", q.popleft(), "| còn:", q)

deque([(0, 1), (1, 1)]) | lấy ra: (0, 0) | còn: deque([(0, 1), (1, 1)])


### 3.3. Stack và DFS lặp

DFS (duyệt theo chiều sâu) dùng **ngăn xếp**: lấy phần tử ở **cuối** (`pop`), thêm vào cuối (`append`). Dùng `list` làm stack thay cho đệ quy tránh `RecursionError` khi đường đi dài.

In [4]:
stack = [(0, 0)]
stack.append((0, 1))
stack.append((1, 1))
print(stack, "| lấy ra:", stack.pop(), "| còn:", stack)

[(0, 0), (0, 1)] | lấy ra: (1, 1) | còn: [(0, 0), (0, 1)]


### 3.4. Mảng `visited` hai chiều và `set`

Để không đi lại một ô, ta ghi nhớ ô đã thăm: hoặc bảng `vis = [[False] * n for _ in range(m)]` (tạo bằng list comprehension, **không** viết `[[False] * n] * m`), hoặc `set` các tọa độ `(r, c)`. Giao của hai tập: `A & B`.

In [5]:
vis = [[False] * 3 for _ in range(2)]
vis[0][1] = True
print(vis)
A = {(0, 0), (0, 1), (1, 1)}
B = {(0, 1), (1, 1), (1, 2)}
print("giao:", sorted(A & B))

[[False, True, False], [False, False, False]]
giao: [(0, 1), (1, 1)]


### 3.5. Sắp xếp danh sách tọa độ để so sánh

Vì thứ tự kết quả không quan trọng, ta chuẩn hóa bằng `sorted(list_of_pairs)`; Python so sánh list từng phần tử một, nên `[0,4] < [1,3]`.

In [6]:
a = [[1, 3], [0, 4], [3, 0]]
b = [[3, 0], [1, 3], [0, 4]]
print(a == b, norm(a) == norm(b), norm(a))

False True [[0, 4], [1, 3], [3, 0]]


## 4. Ý tưởng nền tảng: tính tay

Lấy ví dụ nhỏ `heights = [[1,2],[2,1]]` (hàng 0: `1 2`, hàng 1: `2 1`). Ô sát Thái Bình Dương: hàng `0` và cột `0` ⇒ `(0,0), (0,1), (1,0)`. Ô sát Đại Tây Dương: hàng `1` và cột `1` ⇒ `(0,1), (1,0), (1,1)`.

- `(0,1)`: sát **cả hai** ⇒ nằm trong kết quả. `(1,0)`: sát cả hai ⇒ nằm trong kết quả.
- `(0,0)` (cao `1`): kề Thái Bình Dương. Nước chảy được sang ô kề có độ cao `<= 1`: `(0,1)` cao `2`, `(1,0)` cao `2` — đều cao hơn ⇒ không chảy được. Vậy không tới Đại Tây Dương.
- `(1,1)` (cao `1`): kề Đại Tây Dương; ô kề `(0,1)`, `(1,0)` cao `2` ⇒ không chảy tới Thái Bình Dương.

Kết quả `[[0,1],[1,0]]`.

**Đảo chiều tư duy:** thay vì hỏi "từ ô này nước chảy tới đâu", hỏi "từ **bờ biển**, nước có thể **leo ngược** tới những ô nào". Leo ngược nghĩa là đi sang ô kề có độ cao **lớn hơn hoặc bằng**. Tập ô mà biển Thái Bình Dương "với tới được" chính là tập các ô chảy được tới Thái Bình Dương.

## 5. Cách 1 — Brute Force: đi xuôi dòng nước từ từng ô

**Ý tưởng:** với mỗi ô `(r, c)`, chạy một DFS/BFS **xuôi dòng** (sang ô kề có độ cao `<=`), ghi nhận có chạm mép Thái Bình Dương (`r == 0` hoặc `c == 0`) và mép Đại Tây Dương (`r == m-1` hoặc `c == n-1`) hay không. Nếu chạm cả hai thì ô đó thuộc kết quả.

Để đơn giản và tránh đệ quy sâu, ta dùng stack lặp.

In [7]:
def pacific_atlantic_brute(heights):
    m, n = len(heights), len(heights[0])
    ans = []
    for r in range(m):
        for c in range(n):
            seen = {(r, c)}
            stack = [(r, c)]
            pac = atl = False
            while stack:
                x, y = stack.pop()
                if x == 0 or y == 0:
                    pac = True
                if x == m - 1 or y == n - 1:
                    atl = True
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < m and 0 <= ny < n and (nx, ny) not in seen and heights[nx][ny] <= heights[x][y]:
                        seen.add((nx, ny))
                        stack.append((nx, ny))
            if pac and atl:
                ans.append([r, c])
    return ans


for heights, expected in examples:
    got = pacific_atlantic_brute(heights)
    print(got)
    assert norm(got) == norm(expected)

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
[[0, 0]]


**Độ phức tạp:** mỗi ô chạy một lần duyệt tới `O(m * n)` ô ⇒ Time `O((m * n)^2)`, Space `O(m * n)` cho `seen` và stack.

**Nút thắt:** với `m = n = 200` có `40000` ô, tệ nhất `1.6 * 10^9` thao tác. Mỗi lần xuất phát lại duyệt lại những vùng đã duyệt ở lần trước. Hãy đo thời gian khi lưới có nhiều ô bằng nhau (nước chảy khắp nơi):

In [8]:
for size in [10, 20, 30, 40]:
    flat = [[5] * size for _ in range(size)]
    t = time.perf_counter()
    r = pacific_atlantic_brute(flat)
    dt = time.perf_counter() - t
    print(f"{size}x{size} toàn độ cao bằng nhau: {len(r)} ô, mất {dt * 1000:8.1f} ms")

10x10 toàn độ cao bằng nhau: 100 ô, mất      6.7 ms
20x20 toàn độ cao bằng nhau: 400 ô, mất    106.7 ms


30x30 toàn độ cao bằng nhau: 900 ô, mất    557.5 ms


40x40 toàn độ cao bằng nhau: 1600 ô, mất   1722.5 ms


## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. **Đảo chiều cạnh:** "nước chảy từ `u` sang `v`" khi `h[v] <= h[u]`. Đảo chiều: từ `v` ta "leo ngược" sang `u` khi `h[u] >= h[v]`. Một ô chảy được tới Thái Bình Dương ⇔ **có đường đi leo ngược** từ một ô sát Thái Bình Dương tới ô đó.
2. **Nguồn nhiều điểm:** thay vì chạy từ `m * n` ô, ta chạy **một lần** từ **tất cả ô sát Thái Bình Dương** (đặt hết vào hàng đợi ban đầu — BFS đa nguồn) và **một lần** từ tất cả ô sát Đại Tây Dương. Mỗi ô được thăm tối đa một lần cho mỗi đại dương.

Cuối cùng lấy **giao** hai tập ô thăm được.

Hình dung với ví dụ 1: Thái Bình Dương leo ngược từ hàng 0 và cột 0; Đại Tây Dương leo ngược từ hàng 4 và cột 4. Ô nào ở cả hai bản đồ ⇒ nằm trong kết quả. Đúng cấu trúc ta sẽ in ở mục 8.

## 7. Các cách tiếp theo

### Cách 2 — BFS ngược từ bờ biển, đa nguồn (chốt)

**Giải thích từng đoạn:**

1. Hàm `bfs(sources)`: tạo bảng `vis` toàn `False`, hàng đợi `q` chứa các ô nguồn (đánh dấu `vis` **ngay khi thêm vào hàng đợi**).
2. Lấy `(x, y)` từ đầu hàng đợi; với 4 hướng, nếu ô kề trong biên, chưa thăm, và `heights[nx][ny] >= heights[x][y]` thì đánh dấu thăm và thêm vào hàng đợi.
3. Nguồn Thái Bình Dương: mọi `(i, 0)` và `(0, j)`. Nguồn Đại Tây Dương: mọi `(i, n-1)` và `(m-1, j)`.
4. Kết quả: các ô có `vis1[i][j] and vis2[i][j]`.

In [9]:
def pacific_atlantic_bfs(heights):
    m, n = len(heights), len(heights[0])

    def bfs(sources):
        vis = [[False] * n for _ in range(m)]
        q = deque()
        for x, y in sources:
            if not vis[x][y]:
                vis[x][y] = True
                q.append((x, y))
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                    vis[nx][ny] = True
                    q.append((nx, ny))
        return vis

    pacific = [(i, 0) for i in range(m)] + [(0, j) for j in range(n)]
    atlantic = [(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)]
    v1, v2 = bfs(pacific), bfs(atlantic)
    return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


for heights, expected in examples:
    got = pacific_atlantic_bfs(heights)
    print(got)
    assert norm(got) == norm(expected)

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
[[0, 0]]


In [10]:
# Code nộp LeetCode — Cách 2 (O(m*n) time, O(m*n) space)
from collections import deque


class SolutionBFS:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        m, n = len(heights), len(heights[0])
        dirs = (-1, 0, 1, 0, -1)

        def bfs(q):
            vis = [[False] * n for _ in range(m)]
            for x, y in q:
                vis[x][y] = True
            q = deque(q)
            while q:
                x, y = q.popleft()
                for k in range(4):
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                        vis[nx][ny] = True
                        q.append((nx, ny))
            return vis

        v1 = bfs([(i, 0) for i in range(m)] + [(0, j) for j in range(1, n)])
        v2 = bfs([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n - 1)])
        return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


print([norm(SolutionBFS().pacificAtlantic([r[:] for r in h])) == norm(e) for h, e in examples])

[True, True]


**Độ phức tạp Cách 2:** mỗi ô được đưa vào hàng đợi tối đa **một lần** cho mỗi đại dương, mỗi lần xét 4 ô kề ⇒ Time `O(m * n)`, Space `O(m * n)` cho hai bảng `vis` và hàng đợi.

### Cách 3 — DFS lặp (stack) từ bờ biển

**Ý tưởng:** giống hệt Cách 2 nhưng thay `deque` + `popleft` bằng `list` + `pop` (ngăn xếp). Thứ tự thăm khác (sâu trước) nhưng **tập** ô thăm được vẫn giống hệt vì đều là "mọi ô đến được bằng đường leo ngược". Ta ưu tiên **DFS lặp** thay vì đệ quy vì lưới `200 x 200` có thể có đường đi dài hàng chục nghìn ô và đệ quy Python sẽ báo `RecursionError` (xem mục 13).

In [11]:
def pacific_atlantic_dfs(heights):
    m, n = len(heights), len(heights[0])

    def dfs_iter(sources):
        vis = [[False] * n for _ in range(m)]
        stack = []
        for x, y in sources:
            if not vis[x][y]:
                vis[x][y] = True
                stack.append((x, y))
        while stack:
            x, y = stack.pop()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                    vis[nx][ny] = True
                    stack.append((nx, ny))
        return vis

    v1 = dfs_iter([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
    v2 = dfs_iter([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
    return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


for heights, expected in examples:
    got = pacific_atlantic_dfs(heights)
    print(got)
    assert norm(got) == norm(expected)

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
[[0, 0]]


In [12]:
# Code nộp LeetCode — Cách 3 (DFS lặp, O(m*n) time, O(m*n) space)
class SolutionDFS:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        m, n = len(heights), len(heights[0])

        def reach(sources):
            vis = [[False] * n for _ in range(m)]
            stack = []
            for x, y in sources:
                if not vis[x][y]:
                    vis[x][y] = True
                    stack.append((x, y))
            while stack:
                x, y = stack.pop()
                for nx, ny in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):
                    if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                        vis[nx][ny] = True
                        stack.append((nx, ny))
            return vis

        v1 = reach([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
        v2 = reach([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
        return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


print([norm(SolutionDFS().pacificAtlantic([r[:] for r in h])) == norm(e) for h, e in examples])

[True, True]


**Độ phức tạp Cách 3:** Time `O(m * n)`, Space `O(m * n)`. Stack tối đa chứa `O(m * n)` ô nhưng không bao giờ gây tràn call stack của Python.

### Cách 4 — Quy hoạch động theo độ cao tăng dần (xử lý "cao nguyên")

**Một ý tưởng khác hẳn (không duyệt từ bờ biển):** sắp xếp mọi ô theo độ cao **tăng dần** và quyết định `reach[ô]` (ô đó chảy tới đại dương đang xét) theo:

- Ô nằm sát đại dương ⇒ `True`.
- Ô có **ô kề thấp hơn chặt** đã `True` ⇒ nước chảy xuống đó rồi ra biển ⇒ `True`. (Ô thấp hơn đã được xử lý xong vì ta đi từ thấp lên cao.)
- Ô có **ô kề bằng độ cao** đã `True` ⇒ nước chảy ngang sang đó ⇒ `True`. Nhưng ô cùng độ cao thuộc **cùng nhóm** chưa xử lý xong, nên phải **lan truyền trong nhóm**: một "cao nguyên" các ô cùng độ cao liền kề nhau, nếu một ô trong cao nguyên tới được biển thì cả cao nguyên tới được.

**Giải thích từng đoạn:**

1. Gom ô theo độ cao: `groups[h] = [(r, c), ...]`, duyệt `h` tăng dần.
2. Với mỗi nhóm: tìm các **hạt giống** — ô sát biển hoặc có ô kề **thấp hơn chặt** với `reach = True`.
3. BFS **trong nhóm**: từ hạt giống lan sang ô kề **cùng độ cao** chưa `True`.
4. Làm hai lần (cho hai đại dương), rồi giao.

Cách này `O(mn log(mn))` do sắp xếp (hoặc `O(mn + số độ cao khác nhau)` với bucket), chậm hơn BFS thẳng nhưng cho cái nhìn "quy hoạch động trên thứ tự độ cao".

In [13]:
def pacific_atlantic_dp(heights):
    m, n = len(heights), len(heights[0])
    groups = {}
    for r in range(m):
        for c in range(n):
            groups.setdefault(heights[r][c], []).append((r, c))
    levels = sorted(groups)

    def solve(is_coast):
        reach = [[False] * n for _ in range(m)]
        for h in levels:                                    # từ thấp lên cao
            cells = groups[h]
            q = deque()
            for x, y in cells:
                seed = is_coast(x, y)
                if not seed:
                    for dx, dy in DIRS:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and heights[nx][ny] < h and reach[nx][ny]:
                            seed = True
                            break
                if seed and not reach[x][y]:
                    reach[x][y] = True
                    q.append((x, y))
            while q:                                        # lan truyền trong cao nguyên cùng độ cao
                x, y = q.popleft()
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < m and 0 <= ny < n and heights[nx][ny] == h and not reach[nx][ny]:
                        reach[nx][ny] = True
                        q.append((nx, ny))
        return reach

    pac = solve(lambda x, y: x == 0 or y == 0)
    atl = solve(lambda x, y: x == m - 1 or y == n - 1)
    return [[i, j] for i in range(m) for j in range(n) if pac[i][j] and atl[i][j]]


for heights, expected in examples:
    got = pacific_atlantic_dp(heights)
    print(got)
    assert norm(got) == norm(expected)

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
[[0, 0]]


In [14]:
# Code nộp LeetCode — Cách 4 (O(mn log(mn)) time, O(mn) space)
from collections import deque


class SolutionDP:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        m, n = len(heights), len(heights[0])
        groups = {}
        for r in range(m):
            for c in range(n):
                groups.setdefault(heights[r][c], []).append((r, c))
        levels = sorted(groups)

        def solve(coast):
            reach = [[False] * n for _ in range(m)]
            for h in levels:
                q = deque()
                for x, y in groups[h]:
                    seed = coast(x, y)
                    if not seed:
                        for nx, ny in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):
                            if 0 <= nx < m and 0 <= ny < n and heights[nx][ny] < h and reach[nx][ny]:
                                seed = True
                                break
                    if seed:
                        reach[x][y] = True
                        q.append((x, y))
                while q:
                    x, y = q.popleft()
                    for nx, ny in ((x - 1, y), (x + 1, y), (x, y - 1), (x, y + 1)):
                        if 0 <= nx < m and 0 <= ny < n and heights[nx][ny] == h and not reach[nx][ny]:
                            reach[nx][ny] = True
                            q.append((nx, ny))
            return reach

        pac = solve(lambda x, y: x == 0 or y == 0)
        atl = solve(lambda x, y: x == m - 1 or y == n - 1)
        return [[i, j] for i in range(m) for j in range(n) if pac[i][j] and atl[i][j]]


print([norm(SolutionDP().pacificAtlantic([r[:] for r in h])) == norm(e) for h, e in examples])

[True, True]


**Độ phức tạp Cách 4:** gom nhóm và sắp xếp các độ cao khác nhau `O(mn + k log k)` với `k` là số độ cao khác nhau; mỗi ô xét 4 ô kề vài lần ⇒ tổng `O(mn log(mn))` trong trường hợp xấu. Space `O(mn)`.

## 8. TRACE từng bước

Với `heights = ex1`, kết quả BFS ngược (`#` = nước từ đại dương đó "leo ngược" tới được):

Thái Bình Dương:

```
#####
#####
###..
##...
#....
```

Đại Tây Dương:

```
....#
...##
..###
#####
#####
```

Giao của hai bản đồ (ô `#` ở cả hai): `(0,4)`, `(1,3)`, `(1,4)`, `(2,2)`, `(3,0)`, `(3,1)`, `(4,0)` — đúng đáp án đề. Hàm `trace_bfs` in từng lần lấy ô khỏi hàng đợi, các ô kề được thêm vào và vì sao các ô kề khác bị loại.

In [15]:
def show(vis):
    return "\n".join("".join("#" if v else "." for v in row) for row in vis)


def trace_bfs(heights, ocean="Thái Bình Dương", pacific=True, max_steps=14):
    m, n = len(heights), len(heights[0])
    sources = ([(i, 0) for i in range(m)] + [(0, j) for j in range(1, n)]) if pacific else \
              ([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n - 1)])
    vis = [[False] * n for _ in range(m)]
    q = deque()
    for x, y in sources:
        vis[x][y] = True
        q.append((x, y))
    print(f"BFS ngược từ {ocean}: nguồn ban đầu ({len(sources)} ô) = {sources}")
    step = 0
    while q:
        x, y = q.popleft()
        step += 1
        added, rejected = [], []
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if not (0 <= nx < m and 0 <= ny < n):
                continue
            if vis[nx][ny]:
                rejected.append(f"({nx},{ny}) đã thăm")
            elif heights[nx][ny] >= heights[x][y]:
                vis[nx][ny] = True
                q.append((nx, ny))
                added.append(f"({nx},{ny})h={heights[nx][ny]}")
            else:
                rejected.append(f"({nx},{ny})h={heights[nx][ny]}<{heights[x][y]}")
        if step <= max_steps:
            print(f"bước {step:>2}: lấy ({x},{y}) h={heights[x][y]} | thêm {added} | bỏ {rejected}")
    print(f"... tổng {step} ô được lấy ra khỏi hàng đợi")
    print(show(vis))
    return vis


v_pac = trace_bfs(ex1)

BFS ngược từ Thái Bình Dương: nguồn ban đầu (9 ô) = [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (0, 1), (0, 2), (0, 3), (0, 4)]
bước  1: lấy (0,0) h=1 | thêm [] | bỏ ['(0,1) đã thăm', '(1,0) đã thăm']
bước  2: lấy (1,0) h=3 | thêm [] | bỏ ['(0,0) đã thăm', '(1,1)h=2<3', '(2,0) đã thăm']
bước  3: lấy (2,0) h=2 | thêm ['(2,1)h=4'] | bỏ ['(1,0) đã thăm', '(3,0) đã thăm']
bước  4: lấy (3,0) h=6 | thêm ['(3,1)h=7'] | bỏ ['(2,0) đã thăm', '(4,0) đã thăm']
bước  5: lấy (4,0) h=5 | thêm [] | bỏ ['(3,0) đã thăm', '(4,1)h=1<5']
bước  6: lấy (0,1) h=2 | thêm ['(1,1)h=2'] | bỏ ['(0,2) đã thăm', '(0,0) đã thăm']
bước  7: lấy (0,2) h=2 | thêm ['(1,2)h=3'] | bỏ ['(0,3) đã thăm', '(0,1) đã thăm']
bước  8: lấy (0,3) h=3 | thêm ['(1,3)h=4'] | bỏ ['(0,4) đã thăm', '(0,2) đã thăm']
bước  9: lấy (0,4) h=5 | thêm [] | bỏ ['(1,4)h=4<5', '(0,3) đã thăm']
bước 10: lấy (2,1) h=4 | thêm ['(2,2)h=5'] | bỏ ['(1,1) đã thăm', '(3,1) đã thăm', '(2,0) đã thăm']
bước 11: lấy (3,1) h=7 | thêm [] | bỏ ['(2,1) đã thăm', '(3,

In [16]:
v_atl = trace_bfs(ex1, "Đại Tây Dương", pacific=False, max_steps=6)
both = [[i, j] for i in range(5) for j in range(5) if v_pac[i][j] and v_atl[i][j]]
print("giao hai bản đồ:", both)
assert norm(both) == norm(examples[0][1])

BFS ngược từ Đại Tây Dương: nguồn ban đầu (9 ô) = [(0, 4), (1, 4), (2, 4), (3, 4), (4, 4), (4, 0), (4, 1), (4, 2), (4, 3)]
bước  1: lấy (0,4) h=5 | thêm [] | bỏ ['(1,4) đã thăm', '(0,3)h=3<5']
bước  2: lấy (1,4) h=4 | thêm ['(1,3)h=4'] | bỏ ['(0,4) đã thăm', '(2,4) đã thăm']
bước  3: lấy (2,4) h=1 | thêm ['(2,3)h=3'] | bỏ ['(1,4) đã thăm', '(3,4) đã thăm']
bước  4: lấy (3,4) h=5 | thêm [] | bỏ ['(2,4) đã thăm', '(4,4) đã thăm', '(3,3)h=4<5']
bước  5: lấy (4,4) h=4 | thêm [] | bỏ ['(3,4) đã thăm', '(4,3) đã thăm']
bước  6: lấy (4,0) h=5 | thêm ['(3,0)h=6'] | bỏ ['(4,1) đã thăm']
... tổng 16 ô được lấy ra khỏi hàng đợi
....#
...##
..###
#####
#####
giao hai bản đồ: [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


In [17]:
def trace_dp_pacific(heights):
    """In từng nhóm độ cao của Cách 4 cho Thái Bình Dương: hạt giống và cao nguyên được lan truyền."""
    m, n = len(heights), len(heights[0])
    groups = {}
    for r in range(m):
        for c in range(n):
            groups.setdefault(heights[r][c], []).append((r, c))
    reach = [[False] * n for _ in range(m)]
    for h in sorted(groups):
        seeds, q = [], deque()
        for x, y in groups[h]:
            seed = x == 0 or y == 0
            if not seed:
                seed = any(0 <= x + dx < m and 0 <= y + dy < n and heights[x + dx][y + dy] < h and reach[x + dx][y + dy]
                           for dx, dy in DIRS)
            if seed:
                reach[x][y] = True
                q.append((x, y))
                seeds.append((x, y))
        spread = []
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and heights[nx][ny] == h and not reach[nx][ny]:
                    reach[nx][ny] = True
                    q.append((nx, ny))
                    spread.append((nx, ny))
        print(f"độ cao {h}: các ô {groups[h]} | hạt giống {seeds} | lan thêm {spread}")
    print(show(reach))
    return reach


r = trace_dp_pacific(ex1)
assert r == v_pac

độ cao 1: các ô [(0, 0), (2, 4), (3, 2), (4, 1), (4, 2)] | hạt giống [(0, 0)] | lan thêm []
độ cao 2: các ô [(0, 1), (0, 2), (1, 1), (2, 0), (4, 3)] | hạt giống [(0, 1), (0, 2), (2, 0)] | lan thêm [(1, 1)]
độ cao 3: các ô [(0, 3), (1, 0), (1, 2), (2, 3)] | hạt giống [(0, 3), (1, 0), (1, 2)] | lan thêm []
độ cao 4: các ô [(1, 3), (1, 4), (2, 1), (3, 3), (4, 4)] | hạt giống [(1, 3), (2, 1)] | lan thêm [(1, 4)]
độ cao 5: các ô [(0, 4), (2, 2), (3, 4), (4, 0)] | hạt giống [(0, 4), (2, 2), (4, 0)] | lan thêm []
độ cao 6: các ô [(3, 0)] | hạt giống [(3, 0)] | lan thêm []
độ cao 7: các ô [(3, 1)] | hạt giống [(3, 1)] | lan thêm []
#####
#####
###..
##...
#....


## 9. Code nộp LeetCode

Lời giải chính của notebook là **Cách 2** (BFS ngược đa nguồn): ngắn, `O(m * n)`, không đệ quy.

In [18]:
class Solution:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        m, n = len(heights), len(heights[0])
        dirs = (-1, 0, 1, 0, -1)                          # cặp liên tiếp = một hướng

        def bfs(starts):
            vis = [[False] * n for _ in range(m)]
            q = deque()
            for x, y in starts:
                if not vis[x][y]:
                    vis[x][y] = True                      # đánh dấu ngay khi đưa vào hàng đợi
                    q.append((x, y))
            while q:
                x, y = q.popleft()
                for k in range(4):
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if (0 <= nx < m and 0 <= ny < n and not vis[nx][ny]
                            and heights[nx][ny] >= heights[x][y]):   # leo ngược: cao hơn hoặc bằng
                        vis[nx][ny] = True
                        q.append((nx, ny))
            return vis

        pacific = bfs([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
        atlantic = bfs([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
        return [[i, j] for i in range(m) for j in range(n) if pacific[i][j] and atlantic[i][j]]


for heights, expected in examples:
    got = Solution().pacificAtlantic([r[:] for r in heights])
    print(got)
    assert norm(got) == norm(expected)

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
[[0, 0]]


## 10. Vì sao đúng

1. **Đảo chiều cạnh:** nước chảy được từ `u` đến `v` (kề nhau) ⇔ `h[v] <= h[u]`. Một ô `s` chảy được tới Thái Bình Dương ⇔ tồn tại đường đi `s = u0, u1, ..., uk` với `h[u_{i+1}] <= h[u_i]` và `uk` sát Thái Bình Dương. Đảo thứ tự đường đi: từ `uk` đi tới `s` qua các bước `h[u_i] >= h[u_{i+1}]`, tức **leo ngược** (cao hơn hoặc bằng). Vậy tập ô tìm được bởi BFS từ mọi ô sát Thái Bình Dương, đi theo điều kiện "cao hơn hoặc bằng", **chính xác** là tập ô chảy được tới Thái Bình Dương.
2. **Đa nguồn:** ô nguồn đã thỏa trực tiếp (đường đi dài 0); BFS đa nguồn thăm đúng hợp các tập tới được từ từng nguồn.
3. **Giao hai tập:** ô cần tìm chảy được tới **cả hai** đại dương ⇔ thuộc cả hai tập.
4. **Mỗi ô thăm một lần:** đánh dấu `vis` ngay khi đưa vào hàng đợi nên không có ô nào vào hàng đợi hai lần ⇒ `O(m * n)`.
5. **Ô góc:** góc `(0, n-1)` và `(m-1, 0)` thuộc nguồn của cả hai đại dương, nên tự động nằm trong kết quả (ví dụ 2 với lưới `1 x 1`).

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Đi xuôi từ từng ô | `O((m * n)^2)` | `O(m * n)` |
| 2. BFS ngược đa nguồn | **`O(m * n)`** | **`O(m * n)`** |
| 3. DFS lặp (stack) | `O(m * n)` | `O(m * n)` |
| 4. DP theo độ cao + lan truyền cao nguyên | `O(m * n * log(m * n))` | `O(m * n)` |

Với `m = n = 200` Cách 2 xử lý khoảng `40000` ô × 4 hướng × 2 đại dương — chưa tới một giây trong Python.

## 12. Bộ test

Oracle độc lập: `pacific_atlantic_brute` đi **xuôi dòng** từ mỗi ô (mục 5) — ý tưởng ngược hẳn với các cách còn lại. Mọi so sánh dùng `norm` (sắp xếp). Với input lớn đối chiếu giữa các cách nhanh.

In [19]:
SOLUTIONS = [Solution(), SolutionBFS(), SolutionDFS(), SolutionDP()]


def run_all(heights):
    outs = []
    for sol in SOLUTIONS:
        outs.append(norm(sol.pacificAtlantic([r[:] for r in heights])))
    return outs


# 12.1 Ví dụ đề + edge case (so với oracle vét cạn)
cases = [
    ex1,
    [[1]],
    [[1, 2], [2, 1]],
    [[5, 5, 5], [5, 5, 5], [5, 5, 5]],                  # cao nguyên: mọi ô đều chảy được tới cả hai
    [[1], [2], [3], [4]],                               # một cột
    [[1, 2, 3, 4]],                                     # một hàng
    [[4], [3], [2], [1]],
    [[3, 3, 3], [3, 1, 3], [3, 3, 3]],                  # hố ở giữa: ô hố không chảy ra được; kiểm tra bằng oracle
    [[1, 2, 3], [8, 9, 4], [7, 6, 5]],                  # xoắn ốc
    [[10, 10, 10], [10, 1, 10], [10, 10, 10]],
    [[0, 0], [0, 0]],                                   # độ cao 0
    [[100000, 0], [0, 100000]],                         # biên giá trị
    [[1, 1], [1, 1]],
]
for heights in cases:
    expected = norm(pacific_atlantic_brute(heights))
    outs = run_all(heights)
    for o in outs:
        assert o == expected, (heights, o, expected)
print("OK:", len(cases), "lưới x", len(SOLUTIONS), "lớp (đã đối chiếu với vét cạn)")
print("ví dụ lưới 1x1:", norm(Solution().pacificAtlantic([[1]])), "| cột 4x1 tăng dần:", norm(Solution().pacificAtlantic([[1], [2], [3], [4]])))

OK: 13 lưới x 4 lớp (đã đối chiếu với vét cạn)
ví dụ lưới 1x1: [[0, 0]] | cột 4x1 tăng dần: [[0, 0], [1, 0], [2, 0], [3, 0]]


In [20]:
# 12.2 Ngẫu nhiên có seed: lưới nhỏ (m, n <= 7), độ cao nhỏ (nhiều ô bằng nhau) và độ cao lớn
random.seed(417)
nonempty = 0
for _ in range(1500):
    m, n = random.randint(1, 7), random.randint(1, 7)
    hi = random.choice([1, 2, 3, 10, 10**5])
    heights = [[random.randint(0, hi) for _ in range(n)] for _ in range(m)]
    expected = norm(pacific_atlantic_brute(heights))
    for o in run_all(heights):
        assert o == expected, (heights, o, expected)
    assert len(set(map(tuple, expected))) == len(expected)          # không có ô trùng
    nonempty += bool(expected)
print("OK: 1500 lưới ngẫu nhiên nhỏ; số lưới có kết quả không rỗng:", nonempty)

OK: 1500 lưới ngẫu nhiên nhỏ; số lưới có kết quả không rỗng: 1500


In [21]:
# 12.3 Lưới lớn đúng ràng buộc 200 x 200: các cách nhanh thống nhất
random.seed(4171)
for hi in [1, 3, 100, 10**5]:
    heights = [[random.randint(0, hi) for _ in range(200)] for _ in range(200)]
    outs = run_all(heights)
    assert all(o == outs[0] for o in outs)
    print(f"200x200, độ cao <= {hi:>6}: {len(outs[0]):>5} ô trong kết quả (4 cách thống nhất)")

# lưới có cấu trúc: núi ở giữa (cao dần về tâm) và thung lũng (thấp dần về tâm)
mountain = [[-(abs(i - 99.5) + abs(j - 99.5)) for j in range(200)] for i in range(200)]
mountain = [[int(v + 400) for v in row] for row in mountain]
valley = [[int(abs(i - 99.5) + abs(j - 99.5)) for j in range(200)] for i in range(200)]
for name, g in [("núi", mountain), ("thung lũng", valley)]:
    outs = run_all(g)
    assert all(o == outs[0] for o in outs)
    print(f"{name}: {len(outs[0])} ô")
# núi: chỉ cần ô sát hai biển ở đúng góc; thung lũng: viền cao nhất, nước chảy dồn về tâm nhưng tâm không ra biển
assert len(run_all(valley)[0]) < len(run_all(mountain)[0])

200x200, độ cao <=      1:   113 ô trong kết quả (4 cách thống nhất)
200x200, độ cao <=      3:    25 ô trong kết quả (4 cách thống nhất)


200x200, độ cao <=    100:     9 ô trong kết quả (4 cách thống nhất)


200x200, độ cao <= 100000:     4 ô trong kết quả (4 cách thống nhất)


núi: 20398 ô
thung lũng: 2 ô


In [22]:
# 12.4 Hiệu năng: 200 x 200, mọi ô bằng nhau (nước chảy mọi nơi) — trường hợp thăm nhiều nhất
flat = [[7] * 200 for _ in range(200)]
t = time.perf_counter()
res = Solution().pacificAtlantic(flat)
dt = time.perf_counter() - t
assert len(res) == 200 * 200 and dt < 5
print(f"200x200 toàn bằng nhau: {len(res)} ô, mất {dt * 1000:.0f} ms")
t = time.perf_counter()
res2 = SolutionDFS().pacificAtlantic(flat)
dt = time.perf_counter() - t
assert norm(res2) == norm(res)
print(f"DFS lặp cùng input: mất {dt * 1000:.0f} ms")

200x200 toàn bằng nhau: 40000 ô, mất 83 ms
DFS lặp cùng input: mất 51 ms


In [23]:
# 12.5 Phản ví dụ: chỉ lấy 4 ô góc làm nguồn, hoặc so sánh sai chiều
def wrong_corners_only(heights):
    m, n = len(heights), len(heights[0])

    def bfs(starts):
        vis = [[False] * n for _ in range(m)]
        q = deque(starts)
        for x, y in starts:
            vis[x][y] = True
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                    vis[nx][ny] = True
                    q.append((nx, ny))
        return vis

    v1 = bfs([(0, 0)])                       # SAI: chỉ lấy một góc làm nguồn Thái Bình Dương
    v2 = bfs([(m - 1, n - 1)])               # SAI: chỉ lấy một góc làm nguồn Đại Tây Dương
    return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


print("chỉ góc:", norm(wrong_corners_only(ex1)), "\nđúng    :", norm(examples[0][1]))
assert norm(wrong_corners_only(ex1)) != norm(examples[0][1])

chỉ góc: [] 
đúng    : [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đi ngược nhưng so sánh sai chiều

In [24]:
def wrong_direction(heights):
    m, n = len(heights), len(heights[0])

    def bfs(starts):
        vis = [[False] * n for _ in range(m)]
        q = deque()
        for x, y in starts:
            vis[x][y] = True
            q.append((x, y))
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] <= heights[x][y]:   # SAI: <=
                    vis[nx][ny] = True
                    q.append((nx, ny))
        return vis

    v1 = bfs([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
    v2 = bfs([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
    return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


got = norm(wrong_direction(ex1))
print("sai:", got)
print("đúng:", norm(Solution().pacificAtlantic(ex1)))
assert got != norm(examples[0][1])

sai: [[0, 0], [0, 1], [0, 2], [0, 3], [0, 4], [1, 1], [1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 2], [4, 0], [4, 1], [4, 2]]
đúng: [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


**Vì sao sai:** khi bắt đầu từ biển và đi **ngược** dòng chảy, ta phải sang ô **cao hơn hoặc bằng**. Dùng `<=` là đang mô phỏng nước chảy xuôi từ bờ vào trong — hoàn toàn khác. **Cách sửa:** `heights[nx][ny] >= heights[x][y]`. Mẹo nhớ: "đi ngược dòng thì leo lên".

### Lỗi 2: chỉ lấy ô góc (hoặc thiếu một mép) làm nguồn

Xem mục 12.5: mọi ô sát đại dương đều là nguồn — **cả hàng** `0` và **cả cột** `0` cho Thái Bình Dương, không phải riêng góc `(0, 0)`. Thiếu nguồn thì thiếu ô tìm thấy.

In [25]:
def wrong_missing_edge(heights):
    m, n = len(heights), len(heights[0])

    def bfs(starts):
        vis = [[False] * n for _ in range(m)]
        q = deque()
        for x, y in starts:
            vis[x][y] = True
            q.append((x, y))
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                    vis[nx][ny] = True
                    q.append((nx, ny))
        return vis

    v1 = bfs([(0, j) for j in range(n)])                           # SAI: quên cột 0
    v2 = bfs([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
    return [[i, j] for i in range(m) for j in range(n) if v1[i][j] and v2[i][j]]


print("thiếu cột 0:", norm(wrong_missing_edge(ex1)))
print("đúng       :", norm(Solution().pacificAtlantic(ex1)))
assert norm(wrong_missing_edge(ex1)) != norm(examples[0][1])

thiếu cột 0: [[0, 4], [1, 3], [1, 4], [2, 2], [3, 1]]
đúng       : [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


**Vì sao sai:** ô `(3, 0)` và `(4, 0)` ở cột `0` kề Thái Bình Dương nhưng không được đưa vào nguồn nên bị bỏ sót. **Cách sửa:** nguồn Thái Bình Dương = hàng `0` ∪ cột `0`; Đại Tây Dương = hàng `m-1` ∪ cột `n-1`.

### Lỗi 3: dùng chung một bảng `visited` cho hai đại dương

In [26]:
def wrong_shared_visited(heights):
    m, n = len(heights), len(heights[0])
    vis = [[False] * n for _ in range(m)]                          # SAI: một bảng dùng chung

    def bfs(starts):
        q = deque()
        for x, y in starts:
            vis[x][y] = True
            q.append((x, y))
        while q:
            x, y = q.popleft()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                    vis[nx][ny] = True
                    q.append((nx, ny))

    bfs([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
    first = [row[:] for row in vis]
    bfs([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
    return [[i, j] for i in range(m) for j in range(n) if first[i][j] and vis[i][j]]


got = norm(wrong_shared_visited(ex1))
print("sai:", got, "| số ô:", len(got), "(đúng là 7)")
assert got != norm(examples[0][1])

sai: [[0, 0], [0, 1], [0, 2], [0, 3], [0, 4], [1, 0], [1, 1], [1, 2], [1, 3], [1, 4], [2, 0], [2, 1], [2, 2], [3, 0], [3, 1], [4, 0]] | số ô: 16 (đúng là 7)


**Vì sao sai:** lần BFS thứ hai thấy các ô đã được lần thứ nhất đánh dấu nên **không đi qua chúng** — mất đường của Đại Tây Dương. Mỗi đại dương cần một bảng `visited` **riêng**. **Cách sửa:** tạo `vis` mới trong mỗi lần gọi `bfs`.

### Lỗi 4: đệ quy DFS gây `RecursionError`

DFS bằng hàm đệ quy trên lưới có đường đi dài sẽ vượt giới hạn đệ quy mặc định (khoảng `1000`). Lưới `100 x 100` toàn độ cao bằng nhau là hợp lệ và buộc DFS đi qua hàng nghìn ô liên tiếp:

In [27]:
def dfs_recursive(heights):
    m, n = len(heights), len(heights[0])

    def go(x, y, vis):
        vis[x][y] = True
        for dx, dy in ((0, 1), (1, 0), (0, -1), (-1, 0)):
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                go(nx, ny, vis)

    vis = [[False] * n for _ in range(m)]
    go(0, 0, vis)
    return vis


flat100 = [[1] * 100 for _ in range(100)]
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
try:
    dfs_recursive(flat100)
    print("không lỗi")
except RecursionError as e:
    print("RecursionError:", str(e)[:60])
assert sys.getrecursionlimit() <= 3000

giới hạn đệ quy mặc định: 1000
RecursionError: maximum recursion depth exceeded


**Vì sao sai:** mỗi lần gọi đệ quy dùng một khung (frame) của call stack; đường đi của DFS có thể dài tới `m * n = 40000`. **Cách sửa:** dùng BFS với `deque`, hoặc DFS lặp với stack tường minh (Cách 2 và Cách 3). Nới `sys.setrecursionlimit` là giải pháp rủi ro (có thể làm Python sập vì tràn stack hệ thống).

### Lỗi 5: đánh dấu `visited` muộn (khi lấy ra khỏi hàng đợi)

Nếu chỉ đánh dấu khi `popleft`, một ô có thể bị **đưa vào hàng đợi nhiều lần** (từ nhiều ô kề). Kết quả vẫn đúng nhưng hàng đợi phình ra và chậm; trong trường hợp xấu có thể gấp 4 lần. Đo thử:

In [28]:
def bfs_late_mark(heights):
    m, n = len(heights), len(heights[0])
    pushes = 0
    vis = [[False] * n for _ in range(m)]
    q = deque([(i, 0) for i in range(m)] + [(0, j) for j in range(1, n)])
    pushes += len(q)
    while q:
        x, y = q.popleft()
        if vis[x][y]:
            continue
        vis[x][y] = True                                           # đánh dấu muộn
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                q.append((nx, ny))
                pushes += 1
    return vis, pushes


def bfs_early_mark(heights):
    m, n = len(heights), len(heights[0])
    pushes = 0
    vis = [[False] * n for _ in range(m)]
    q = deque()
    for x, y in [(i, 0) for i in range(m)] + [(0, j) for j in range(1, n)]:
        vis[x][y] = True
        q.append((x, y))
        pushes += 1
    while q:
        x, y = q.popleft()
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                vis[nx][ny] = True
                q.append((nx, ny))
                pushes += 1
    return vis, pushes


flat = [[3] * 60 for _ in range(60)]
v_late, p_late = bfs_late_mark(flat)
v_early, p_early = bfs_early_mark(flat)
print(f"số lần push: đánh dấu muộn {p_late} | đánh dấu sớm {p_early} (= số ô {60 * 60})")
assert v_late == v_early and p_early == 3600 and p_late > p_early

số lần push: đánh dấu muộn 7199 | đánh dấu sớm 3600 (= số ô 3600)


**Vì sao tệ hơn:** ô bị đẩy nhiều lần bởi nhiều hàng xóm trước khi được đánh dấu. **Cách sửa:** đánh dấu `vis[nx][ny] = True` **ngay khi** `append` vào hàng đợi.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "ô nào **đến được** biên/đích theo luật di chuyển một chiều" trên lưới ⇒ **đảo chiều**, chạy **BFS/DFS đa nguồn** từ đích rồi lấy giao/hợp. Cùng họ: *Surrounded Regions* (130, tìm ô `O` thoát được ra biên), *Walls and Gates* (286), *Number of Enclaves* (1020), *Rotting Oranges* (994, BFS đa nguồn), *01 Matrix* (542).
- **BFS đa nguồn:** cho **tất cả** nguồn vào hàng đợi **trước** khi bắt đầu lặp.
- **Đảo chiều điều kiện:** nước chảy xuống (`<=`) ⇒ leo ngược lên (`>=`).
- **Hai đại dương ⇒ hai bảng `visited` riêng**, rồi giao.
- **Tránh đệ quy** khi lưới có thể dài `~ 40000` ô trong Python; dùng `deque`/stack lặp.
- **Kiểm chứng:** vét cạn đi xuôi từ từng ô trên lưới nhỏ (mục 12).

## 15. Biến thể

1. **Chỉ một đại dương:** trả về ô chảy tới Thái Bình Dương — một lần BFS.
2. **Nước chảy được cả 8 hướng:** thêm 4 hướng chéo vào `DIRS`.
3. **Nước chỉ chảy sang ô thấp hơn *chặt chẽ*:** đổi `>=` thành `>` khi leo ngược; cao nguyên không còn thông nhau.
4. **Đếm số ô** thay vì liệt kê: `sum` trên giao hai bảng.
5. **Đảo tên "Đại dương" là các mép tuỳ ý** (ví dụ chỉ cạnh trên và cạnh dưới): chỉ đổi danh sách nguồn.

In [29]:
def reach_one_ocean(heights, sources, strict=False, eight_dirs=False):
    """BFS ngược tổng quát: strict=True -> chỉ sang ô cao hơn chặt chẽ; eight_dirs=True -> thêm đường chéo."""
    m, n = len(heights), len(heights[0])
    dirs = DIRS + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if eight_dirs else [])
    vis = [[False] * n for _ in range(m)]
    q = deque()
    for x, y in sources:
        if not vis[x][y]:
            vis[x][y] = True
            q.append((x, y))
    while q:
        x, y = q.popleft()
        for dx, dy in dirs:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny]:
                ok = heights[nx][ny] > heights[x][y] if strict else heights[nx][ny] >= heights[x][y]
                if ok:
                    vis[nx][ny] = True
                    q.append((nx, ny))
    return vis


m, n = 5, 5
pac_src = [(i, 0) for i in range(m)] + [(0, j) for j in range(n)]
v_normal = reach_one_ocean(ex1, pac_src)
assert v_normal == v_pac
print("chỉ Thái Bình Dương: số ô chảy tới =", sum(map(sum, v_normal)))
v_strict = reach_one_ocean(ex1, pac_src, strict=True)
v_eight = reach_one_ocean(ex1, pac_src, eight_dirs=True)
print("strict:", sum(map(sum, v_strict)), "ô | 8 hướng:", sum(map(sum, v_eight)), "ô")
assert sum(map(sum, v_strict)) <= sum(map(sum, v_normal)) <= sum(map(sum, v_eight))

chỉ Thái Bình Dương: số ô chảy tới = 16
strict: 14 ô | 8 hướng: 20 ô


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi xuôi từ từng ô | mỗi ô chạy DFS xem chạm hai mép | `O((m * n)^2)` | `O(m * n)` | **oracle** trên lưới nhỏ |
| 2. BFS ngược đa nguồn | từ mọi ô sát biển leo ngược lên, lấy giao | **`O(m * n)`** | `O(m * n)` | **nộp bài**, phỏng vấn |
| 3. DFS lặp (stack) | như 2 nhưng dùng stack | `O(m * n)` | `O(m * n)` | thích DFS, tránh đệ quy |
| 4. DP theo độ cao | xử lý ô từ thấp lên cao, lan truyền cao nguyên | `O(m * n * log(m * n))` | `O(m * n)` | góc nhìn khác, kiểm chứng chéo |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `sorted([[1, 3], [0, 4], [1, 1]])`; (b) `{(0, 1), (1, 1)} & {(1, 1), (2, 2)}`; (c) `Solution().pacificAtlantic([[1]])`; (d) với `heights = [[1, 2]]`, kết quả của `Solution().pacificAtlantic(heights)`.

**Bài 2 (tính tay).** Cho `heights = [[1, 2, 3], [8, 9, 4], [7, 6, 5]]`. Hãy viết bản đồ `#`/`.` cho Thái Bình Dương (nguồn hàng `0` và cột `0`) và cho Đại Tây Dương (nguồn hàng `2` và cột `2`) rồi lấy giao.

**Bài 3 (điền TODO).** Hoàn thành `reach_todo(heights, sources)`: trả về bảng `vis` các ô tới được bằng BFS ngược từ `sources`.

In [30]:
def reach_todo(heights, sources):
    # TODO 1: m, n = len(heights), len(heights[0]); vis = [[False] * n for _ in range(m)]
    # TODO 2: q = deque(); với mỗi (x, y) trong sources: đánh dấu vis và append vào q
    # TODO 3: while q: x, y = q.popleft(); với 4 hướng: nếu trong biên, chưa thăm, heights[nx][ny] >= heights[x][y]:
    #         đánh dấu vis và append
    # TODO 4: return vis
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `pacific_atlantic_dfs2(heights)` bằng **DFS lặp** (stack) **không nhìn lại** mục 7. Kiểm tra bằng `norm(...)` với ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `count_flow_to_both(heights)` trả về **chỉ số lượng** ô chảy tới cả hai đại dương, nhưng với luật mới: nước chỉ chảy sang ô thấp hơn **chặt chẽ** (`<`, không còn chảy ngang qua ô bằng nhau). Kiểm tra bằng vét cạn xuôi dòng với luật `<` trên các lưới ngẫu nhiên nhỏ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[[0, 4], [1, 1], [1, 3]]`; `{(1, 1)}`; `[[0, 0]]`; `[[0, 0], [0, 1]]` (hai ô đều sát cả hai biển vì lưới chỉ có một hàng). Bài 3: xem `bfs` ở mục 9.
- **Mức 2 (vừa):** Bài 2: Thái Bình Dương bắt đầu từ hàng 0 (`1 2 3`) và cột 0 (`1 8 7`): từ `3` leo xuống `(1,2) = 4` rồi `(2,2) = 5` rồi `(2,1) = 6`; từ `8` leo sang `9`. Cuối cùng cả 9 ô đều tới được. Đại Tây Dương bắt đầu từ hàng 2 và cột 2, leo ngược được tới 7 ô (trừ `(0,0)` và `(0,1)`). Giao là 7 ô; hãy tự trace rồi so với bản đồ do hàm `show` in ở mục 19. Bài 4: stack + `pop()`.
- **Mức 3 (sâu):** Bài 5: leo ngược với luật mới "nước chảy sang ô **thấp hơn chặt**" nghĩa là ngược lại phải sang ô **cao hơn chặt** (`>`); sau đó đối chiếu với `brute_strict` đi xuôi bằng `<`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert sorted([[1, 3], [0, 4], [1, 1]]) == [[0, 4], [1, 1], [1, 3]]
assert {(0, 1), (1, 1)} & {(1, 1), (2, 2)} == {(1, 1)}
assert Solution().pacificAtlantic([[1]]) == [[0, 0]]
assert norm(Solution().pacificAtlantic([[1, 2]])) == [[0, 0], [0, 1]]

# Bài 2
g = [[1, 2, 3], [8, 9, 4], [7, 6, 5]]
pac = reach_one_ocean(g, [(i, 0) for i in range(3)] + [(0, j) for j in range(3)])
atl = reach_one_ocean(g, [(i, 2) for i in range(3)] + [(2, j) for j in range(3)])
print("Thái Bình Dương:\n" + show(pac))
print("Đại Tây Dương:\n" + show(atl))
both = [[i, j] for i in range(3) for j in range(3) if pac[i][j] and atl[i][j]]
assert norm(both) == norm(Solution().pacificAtlantic(g)) == norm(pacific_atlantic_brute(g))
print("giao:", both)


# Bài 3
def reach_todo_ans(heights, sources):
    m, n = len(heights), len(heights[0])
    vis = [[False] * n for _ in range(m)]
    q = deque()
    for x, y in sources:
        if not vis[x][y]:
            vis[x][y] = True
            q.append((x, y))
    while q:
        x, y = q.popleft()
        for dx, dy in DIRS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                vis[nx][ny] = True
                q.append((nx, ny))
    return vis


assert reach_todo_ans(g, [(i, 0) for i in range(3)] + [(0, j) for j in range(3)]) == pac


# Bài 4
def pacific_atlantic_dfs2(heights):
    m, n = len(heights), len(heights[0])

    def walk(starts):
        vis = [[False] * n for _ in range(m)]
        st = []
        for s in starts:
            if not vis[s[0]][s[1]]:
                vis[s[0]][s[1]] = True
                st.append(s)
        while st:
            x, y = st.pop()
            for dx, dy in DIRS:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                    vis[nx][ny] = True
                    st.append((nx, ny))
        return vis

    a = walk([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
    b = walk([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
    return [[i, j] for i in range(m) for j in range(n) if a[i][j] and b[i][j]]


for h, e in examples:
    assert norm(pacific_atlantic_dfs2(h)) == norm(e)


# Bài 5
def count_flow_to_both(heights):
    m, n = len(heights), len(heights[0])
    pac = reach_one_ocean(heights, [(i, 0) for i in range(m)] + [(0, j) for j in range(n)], strict=True)
    atl = reach_one_ocean(heights, [(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)], strict=True)
    return sum(1 for i in range(m) for j in range(n) if pac[i][j] and atl[i][j])


def brute_strict(heights):
    m, n = len(heights), len(heights[0])
    total = 0
    for r in range(m):
        for c in range(n):
            seen, st = {(r, c)}, [(r, c)]
            pac = atl = False
            while st:
                x, y = st.pop()
                pac |= x == 0 or y == 0
                atl |= x == m - 1 or y == n - 1
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < m and 0 <= ny < n and (nx, ny) not in seen and heights[nx][ny] < heights[x][y]:
                        seen.add((nx, ny))
                        st.append((nx, ny))
            total += pac and atl
    return total


random.seed(4172)
for _ in range(400):
    m, n = random.randint(1, 6), random.randint(1, 6)
    h = [[random.randint(0, 4) for _ in range(n)] for _ in range(m)]
    assert count_flow_to_both(h) == brute_strict(h), h
print("OK: toàn bộ đáp án đúng")

Thái Bình Dương:
###
###
###
Đại Tây Dương:
..#
###
###
giao: [[0, 2], [1, 0], [1, 1], [1, 2], [2, 0], [2, 1], [2, 2]]
OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (BFS ngược đa nguồn, hai đại dương, lấy giao)

```python
class Solution:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        m, n = len(heights), len(heights[0])
        dirs = (-1, 0, 1, 0, -1)

        def bfs(starts):
            vis = [[False] * n for _ in range(m)]
            q = deque()
            for x, y in starts:
                if not vis[x][y]:
                    vis[x][y] = True
                    q.append((x, y))
            while q:
                x, y = q.popleft()
                for k in range(4):
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if 0 <= nx < m and 0 <= ny < n and not vis[nx][ny] and heights[nx][ny] >= heights[x][y]:
                        vis[nx][ny] = True
                        q.append((nx, ny))
            return vis

        p = bfs([(i, 0) for i in range(m)] + [(0, j) for j in range(n)])
        a = bfs([(i, n - 1) for i in range(m)] + [(m - 1, j) for j in range(n)])
        return [[i, j] for i in range(m) for j in range(n) if p[i][j] and a[i][j]]
```

**Checklist**

- [ ] Đảo chiều: từ **biển** đi sang ô **cao hơn hoặc bằng** (`>=`)?
- [ ] Nguồn Thái Bình Dương = **hàng 0 và cột 0**; Đại Tây Dương = **hàng `m-1` và cột `n-1`** (cả hai đều đủ, không chỉ góc)?
- [ ] Hai bảng `visited` **riêng biệt**, rồi lấy giao?
- [ ] Đánh dấu `visited` **ngay khi đưa vào hàng đợi**; kiểm tra biên trước khi truy cập?
- [ ] Không dùng đệ quy sâu (dùng `deque` hoặc stack lặp)?
- [ ] Đã thử lưới `1 x 1`, một hàng, một cột, toàn bằng nhau, lưới `200 x 200`; so sánh sau khi `sorted`?